# 🐍 Day 7: Mini-Project — Word Counter CLI

**Topics covered:**
- Building a complete CLI word counter tool
- Count words, lines, characters
- argparse for options
- File handling and logging

## Step 1: Define the Counter Logic

Functions to count words, lines, and characters in text.

In [ ]:
def count_words(text):
    """Return total word count."""
    return len(text.split())

def count_lines(text):
    """Return total line count."""
    return len(text.splitlines())

def count_chars(text):
    """Return total character count."""
    return len(text)

def count_all(text):
    """Return dict with words, lines, chars."""
    return {
        "words": count_words(text),
        "lines": count_lines(text),
        "chars": count_chars(text)
    }

sample = "Hello world\nThis is a test\n"
print(count_all(sample))

# Ensure sample.txt exists for later cells
from pathlib import Path
Path("sample.txt").write_text("Hello world\nThis is a test\nLine three", encoding="utf-8")

## Step 2: Read from File or stdin

Support both file path and reading from stdin (for piping).

In [ ]:
import sys

def read_input(path=None):
    """Read from file or stdin. Returns (text, source_name)."""
    if path is None or path == "-":
        return sys.stdin.read(), "<stdin>"
    with open(path, encoding="utf-8") as f:
        return f.read(), path

text, name = read_input("sample.txt")
print(f"Read from {name}: {len(text)} chars")

## Step 3: Format Output

Display results in a readable format. Support optional JSON output.

In [ ]:
import json

def format_output(stats, source, json_format=False):
    """Format stats for display."""
    if json_format:
        return json.dumps({"source": source, **stats})
    lines = [f"  {k}: {v}" for k, v in stats.items()]
    return f"{source}:\n" + "\n".join(lines)

stats = count_all("Hello world\nLine 2")
print(format_output(stats, "test.txt"))
print("---")
print(format_output(stats, "test.txt", json_format=True))

## Step 4: Argument Parsing

Use argparse for -w (words), -l (lines), -c (chars), -j (json), and file path.

In [ ]:
import argparse

def parse_args(argv=None):
    parser = argparse.ArgumentParser(description="Count words, lines, chars in files")
    parser.add_argument("files", nargs="*", default=["-"], help="Files to process (default: stdin)")
    parser.add_argument("-w", "--words", action="store_true", help="Count words")
    parser.add_argument("-l", "--lines", action="store_true", help="Count lines")
    parser.add_argument("-c", "--chars", action="store_true", help="Count characters")
    parser.add_argument("-j", "--json", action="store_true", help="JSON output")
    args = parser.parse_args(argv)
    if not (args.words or args.lines or args.chars):
        args.words = args.lines = args.chars = True
    return args

## Step 5: Main Logic

Process each file, compute requested stats, and print.

In [ ]:
def process_file(path, args):
    try:
        text, source = read_input(path)
    except FileNotFoundError:
        print(f"Error: File not found: {path}", file=sys.stderr)
        return None
    except Exception as e:
        print(f"Error: {e}", file=sys.stderr)
        return None
    
    stats = {}
    if args.words:
        stats["words"] = count_words(text)
    if args.lines:
        stats["lines"] = count_lines(text)
    if args.chars:
        stats["chars"] = count_chars(text)
    
    return stats, source

def main():
    args = parse_args()
    for path in args.files:
        result = process_file(path, args)
        if result:
            stats, source = result
            print(format_output(stats, source, json_format=args.json))

# Simulate: wc -w -l sample.txt
args = parse_args(["sample.txt"])
args.words = args.lines = args.chars = True
args.json = False
result = process_file("sample.txt", args)
if result:
    print(format_output(result[0], result[1], args.json))

## Step 6: Complete Standalone Script

Full script with proper argparse and main entry.

In [ ]:
#!/usr/bin/env python3
"""
Word Counter CLI - Count words, lines, and characters in files.
Usage: python word_counter.py [options] [files...]
       echo 'hello world' | python word_counter.py -
"""

import argparse
import json
import sys

def count_words(text):
    return len(text.split())

def count_lines(text):
    return len(text.splitlines())

def count_chars(text):
    return len(text)

def read_input(path=None):
    if path is None or path == "-":
        return sys.stdin.read(), "<stdin>"
    with open(path, encoding="utf-8") as f:
        return f.read(), path

def format_output(stats, source, json_format=False):
    if json_format:
        return json.dumps({"source": source, **stats})
    lines = [f"  {k}: {v}" for k, v in stats.items()]
    return f"{source}:\n" + "\n".join(lines)

def main():
    parser = argparse.ArgumentParser(description="Count words, lines, chars")
    parser.add_argument("files", nargs="*", default=["-"], help="Files (default: stdin)")
    parser.add_argument("-w", "--words", action="store_true")
    parser.add_argument("-l", "--lines", action="store_true")
    parser.add_argument("-c", "--chars", action="store_true")
    parser.add_argument("-j", "--json", action="store_true")
    args = parser.parse_args()  # Uses sys.argv when run as script
    if not (args.words or args.lines or args.chars):
        args.words = args.lines = args.chars = True
    for path in args.files:
        try:
            text, source = read_input(path)
        except FileNotFoundError:
            print(f"Error: {path} not found", file=sys.stderr)
            continue
        stats = {}
        if args.words: stats["words"] = count_words(text)
        if args.lines: stats["lines"] = count_lines(text)
        if args.chars: stats["chars"] = count_chars(text)
        print(format_output(stats, source, args.json))

if __name__ == "__main__":
    main()

## Common Mistakes

1. **Forgetting encoding**: Use `encoding='utf-8'` for text files
2. **stdin in notebook**: Use file paths when testing in notebooks
3. **parse_args() with no args**: In scripts, it uses sys.argv; in notebooks, pass a list
4. **No error handling**: Add try/except for FileNotFoundError and IOError

## Exercises (Enhancements)

Try these enhancements:

### Exercise 1: Add a --bytes option to count bytes (len(text.encode())).

In [ ]:
# YOUR CODE HERE
pass

### Exercise 2: Add a --total flag that prints a summary line when multiple files are given.

In [ ]:
# YOUR CODE HERE
pass

### Exercise 3: Use logging for debug messages when -v/--verbose is set.

In [ ]:
# YOUR CODE HERE
pass

## Key Takeaways

- Split logic: count functions, read_input, format_output, main
- argparse for -w, -l, -c, -j, and file list
- Support stdin with path="-" and sys.stdin.read()
- Handle FileNotFoundError and use encoding for text files

## Next Month Teaser

You've completed Month 1 Foundations! Next we'll dive into **Object-Oriented Programming** — classes, inheritance, magic methods, and design patterns. Stay tuned!